In [5]:
from utils import SPLITS, MODELS_FOLDER, LANGUAGES
from sick import SICKMergedDataset
from activations import ActivationDataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch as t

device: t.device = t.device("cuda" if t.cuda.is_available() else "cpu")

Check the total number of elements in the sick merged dataset

In [6]:
for language in LANGUAGES:
    number_of_elements = 0
    amount_per_split: dict[str, int] = {split: 0 for split in SPLITS}

    for split in SPLITS:
        sick_merged_dataset = SICKMergedDataset(language, split)
        for sick_merged_element in sick_merged_dataset:
            # print(sick_merged_element, number_of_elements)
            # all_sick_merged_elements.append(sick_merged_element)
            amount_per_split[split] += 1

    print("--------")
    print(f"Language={language}")
    print(f"amount_per_split= {amount_per_split}")
    print(f"Total amount of items: {sum(amount_per_split.values())}")

--------
Language=en
amount_per_split= {'train': 4439, 'test': 4906, 'val': 495}
Total amount of items: 9840
--------
Language=es
amount_per_split= {'train': 4439, 'test': 4906, 'val': 495}
Total amount of items: 9840
--------
Language=jp
amount_per_split= {'train': 4439, 'test': 4906, 'val': 495}
Total amount of items: 9840
--------
Language=nl
amount_per_split= {'train': 4439, 'test': 4906, 'val': 495}
Total amount of items: 9840


Check that the activations make sense

In [7]:
activations_dataset = ActivationDataset("es", "train", 0, "standard", "olmo_model")

for i in range(8):
    activations, labels = activations_dataset[i]
    print(f"{activations}\n\n{labels}\n-----------------")

tensor([-0.0072, -0.0162,  0.0128,  ..., -0.0142, -0.0079,  0.0043],
       dtype=torch.bfloat16)

1
-----------------
tensor([-0.0071, -0.0162,  0.0126,  ..., -0.0145, -0.0081,  0.0044],
       dtype=torch.bfloat16)

1
-----------------
tensor([-0.0079, -0.0164,  0.0123,  ..., -0.0132, -0.0074,  0.0043],
       dtype=torch.bfloat16)

0
-----------------
tensor([-0.0074, -0.0161,  0.0128,  ..., -0.0138, -0.0079,  0.0042],
       dtype=torch.bfloat16)

1
-----------------
tensor([-0.0074, -0.0162,  0.0127,  ..., -0.0135, -0.0078,  0.0042],
       dtype=torch.bfloat16)

1
-----------------
tensor([-0.0093, -0.0170,  0.0131,  ..., -0.0135, -0.0071,  0.0040],
       dtype=torch.bfloat16)

1
-----------------
tensor([-0.0091, -0.0162,  0.0134,  ..., -0.0143, -0.0082,  0.0036],
       dtype=torch.bfloat16)

1
-----------------
tensor([-0.0085, -0.0162,  0.0130,  ..., -0.0137, -0.0082,  0.0040],
       dtype=torch.bfloat16)

1
-----------------


Check the sentences in batch 26, which seem to cause memory issues

In [ ]:
from sick import SICKMergedDataset

language = "es"
split = "train"
batch_size = 128

sick_merged_dataset = SICKMergedDataset(language, split)

def find_longest_sentence(skipped_batches=[]):
    longest_sentence = ""
    len_longest_sentence = 0
    i_longest_sentence = 0

    i = 0
    for sentences, _, _ in sick_merged_dataset:
        batch = i // batch_size
        if batch not in skipped_batches:
            for sentence in sentences:
                len_sentence = len(sentence)
                if len_sentence > len_longest_sentence:
                    longest_sentence = sentence
                    len_longest_sentence = len_sentence
                    i_longest_sentence = i

        i += 1

    print(f"Longest sentence:")
    print(longest_sentence)
    print(f"At i = {i_longest_sentence} (batch {i_longest_sentence // batch_size})")
    print(f"With length {len_longest_sentence}")
    # print(f"{sentence_a}\t{sentence_b}")
    # print(len(sentence_a), len(sentence_b))
    # print("-"*20)

find_longest_sentence()

Check model inference

In [ ]:
def run_nli_inference(tokenizer, hf_model):
    """Run NLI task inference with any model."""
    sentence_tuple_batch = [
        (
            "Three boys are jumping in the leaves",
            "Three kids are jumping in the leaves",
        ),
        (
            "Two women are sparring in a kickboxing match",
            "Two people are kickboxing and spectators are not watching",
        ),
    ]

    prompts = [
        f"Premise: {sent_a}. Hypothesis: {sent_b}. Do these sentences entail, contradict, or are neutral to each other?"
        for sent_a, sent_b in sentence_tuple_batch
    ]

    tokens = tokenizer(prompts, return_tensors="pt", padding=True).to(device)

    with t.no_grad():
        generated_ids = hf_model.generate(
            **tokens, max_new_tokens=20, pad_token_id=tokenizer.pad_token_id
        )

    gen_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

    for i, text in enumerate(gen_text):
        print(f"Response {i+1}: {text}")

In [ ]:
# model_name: str = "olmo_model"
model_name: str = "tiny_aya_global"
model_filepath: str = f"{MODELS_FOLDER}/{model_name}"

tokenizer = AutoTokenizer.from_pretrained(
    model_filepath,
    local_files_only=True,
    #   padding_side='left',
)
hf_model = AutoModelForCausalLM.from_pretrained(
    model_filepath, local_files_only=True
).to(device)  # type: ignore

In [ ]:
run_nli_inference(tokenizer, hf_model)